In [0]:
# Path to the source CSV file containing raw diagnosis data
source_path = 'abfss://data@apexlife.dfs.core.windows.net/staging/visits/'

In [0]:
# Path for storing streaming checkpoint information
checkpoint_path = "abfss://data@apexlife.dfs.core.windows.net/bronze/visits_raw/checkpoint/"

# Path for storing schema information inferred from the data
schema_location = "abfss://data@apexlife.dfs.core.windows.net/bronze/visits_raw/schema/"

In [0]:
# Read streaming CSV files from cloud storage using Auto Loader

df = (
    spark.readStream.format('cloudFiles')                  # Use Auto Loader for efficient file discovery
    .option("cloudFiles.format", "csv")                    # Specify the file format as CSV
    .option("header", "true")                              # Indicate that CSV files contain a header row
    .option("inferSchema", "true")                         # Automatically infer the schema from the data
    .option("cloudFiles.schemaLocation", schema_location)  # Path to store the inferred schema
    .option("cloudFiles.maxFilesPerTrigger", 1)            # Process one file per trigger for controlled ingestion
    .load(source_path)  
)

In [0]:
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")            
    .trigger(availableNow=True)
    .table("apexlife.bronze.visits_raw")
)

In [0]:
%sql
select count(*) from apexlife.bronze.visits_raw

count(*)
8
